# Sentiment Analysis using NLP Pipeline & Machine Learning

## Objective
The objective of this project is to build an end-to-end sentiment analysis system using Natural Language Processing (NLP) techniques and Machine Learning models.

We perform:
- Text preprocessing
- Feature engineering (BoW & TF-IDF)
- Model training (Logistic Regression, Naive Bayes, Decision Tree)
- Performance evaluation and comparison

---

In [12]:
import pandas as pd
import numpy as np
import re

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Download resources
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Load Dataset

In [13]:
df = pd.read_csv('/home/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [14]:
print("Dataset Shape:", df.shape)
print("\nClass Distribution:\n", df['sentiment'].value_counts())

Dataset Shape: (50000, 2)

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


## Data Understanding

- The dataset contains movie reviews labeled as positive or negative.
- It is a balanced dataset.
- Each row consists of:
  - Review text
  - Sentiment label

Example review:

In [15]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

## NLP Preprocessing Steps

- Lowercasing
- Removing URLs
- Removing special characters
- Tokenization
- Stopword removal
- Lemmatization

In [16]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

In [17]:
df['cleaned_review'] = df['review'].apply(preprocess_text)
df[['review', 'cleaned_review']].head()

,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode hoo...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


## Feature Engineering

We convert text into numerical vectors using:
- Bag of Words (BoW)
- TF-IDF

In [18]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_review']).toarray()

In [19]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_review']).toarray()

In [20]:
y = df['sentiment'].map({'positive': 1, 'negative': 0})

In [21]:
## Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [22]:
## Model Training
# Logistic Regression


lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [23]:
# Naive Bayes

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

In [24]:
# Decision Tree

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [25]:
# Model Evaluation

def evaluate(y_test, y_pred, model_name):
    print(f"===== {model_name} =====")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("\n")

In [26]:
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")

===== Logistic Regression =====
Accuracy : 0.8893
Precision: 0.8798299845440495
Recall   : 0.9037507441952768
F1 Score : 0.8916299559471366


===== Naive Bayes =====
Accuracy : 0.856
Precision: 0.8526357044875563
Recall   : 0.8634649732089701
F1 Score : 0.8580161703805955


===== Decision Tree =====
Accuracy : 0.7187
Precision: 0.721272365805169
Recall   : 0.719984123834094
F1 Score : 0.7206276690833251




In [27]:
## Comparison Table

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_dt)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_nb),
        recall_score(y_test, y_pred_dt)
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_dt)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8893,0.879830,0.903751,0.891630
1,Naive Bayes,0.8560,0.852636,0.863465,0.858016
2,Decision Tree,0.7187,0.721272,0.719984,0.720628


## Insights & Conclusion

### 🔍 Observations:
- Logistic Regression performed the best due to its efficiency in handling high-dimensional sparse data.
- Naive Bayes performed well and was computationally efficient but slightly less accurate.
- Decision Tree showed lower performance due to overfitting on text features.

### ⚙️ Feature Engineering:
- TF-IDF outperformed Bag of Words as it gives importance to meaningful words and reduces noise.

### 🧹 Preprocessing Impact:
- Stopword removal reduced noise.
- Lemmatization improved word normalization.
- Cleaning special characters improved consistency.

### ⚖️ Trade-offs:
- Logistic Regression → Best accuracy, moderate computation
- Naive Bayes → Fast, good baseline
- Decision Tree → Less suitable for text data

### ✅ Conclusion:
An effective NLP pipeline significantly improves model performance. Among the tested models, Logistic Regression with TF-IDF features provided the best results.